# COCO neural channel — resumable, batched

29 reviewed decoders. Same production bank as digital; five full messages and Native / Matched 39, 42, 45. **No new embedding.**

Upload the matching source ZIP; code is readable Python. Start with **5 images, A100 high RAM**. Do not interrupt the digital experiment or start both in one runtime.


## 0. Configuration

Change only `N_IMAGES` from 5 to 100 after reviewing the canary. Keep the same output folder and batch size.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile

N_IMAGES = 5                  # After the canary: change ONLY this to 100.
MODELS = "all"                # All 29 reviewed production entries.
GPU_BATCH_SIZE = 4            # Freeze this after the first canary; A100 high RAM recommended.
MAX_HOURS = 16                # Session budget; not a guarantee of Colab uptime.
SSD_RESERVE_GIB = 60
CAPTIONS_FILE = None          # Optional official captions_val2017.json; otherwise downloaded once.
SOURCE_ARCHIVE = Path("COCO_Neural_Channel_RESUMABLE_source.zip")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if os.name != "posix":
    raise RuntimeError("Use Colab or Linux/NVIDIA; the complete release cannot run natively on Windows.")
if IN_COLAB:
    drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path("/workspace/data")
BANK_ROOT = BASE / "watermarking_production_bank_v1"   # READ ONLY
OUTPUT_ROOT = BASE / "watermarking_neural_channel_v1" # Separate from digital!
SCRATCH_ROOT = Path("/content/wmbench_neural_scratch") if IN_COLAB else Path("/workspace/wmbench_neural_scratch")
SETTINGS = dict(n_images=N_IMAGES, models=MODELS, gpu_batch_size=GPU_BATCH_SIZE, max_hours=MAX_HOURS,
    ssd_reserve_gib=SSD_RESERVE_GIB, device="cuda", bank=str(BANK_ROOT), output=str(OUTPUT_ROOT),
    scratch=str(SCRATCH_ROOT), captions_file=CAPTIONS_FILE, pairwise=True, download_author_assets=True,
    manual_assets_root=str(BANK_ROOT / "_manual_assets"))
print("Input bank (unchanged):", BANK_ROOT)
print("Durable results:", OUTPUT_ROOT)
print("Temporary attack ZIPs on Drive: one method/pipeline stage, deleted after decoding.")
print("Do not use the runtime that is currently executing the digital experiment.")


## 1. Restore the release and isolated controller

No model weights are stored in this small ZIP. Checkpoints download once per Colab session and are shared between methods. Compatible neural pipelines use the controller environment; each watermark decoder uses its own frozen environment.


In [ ]:
EXPECTED_SHA = "5b70653cf2fb7b23d6ff1b8585aa1b684808ba311d956cf5fc20da4b6a6311e2"
SAVED_ZIP = OUTPUT_ROOT / "_release" / SOURCE_ARCHIVE.name
if not SOURCE_ARCHIVE.is_file() and SAVED_ZIP.is_file():
    SOURCE_ARCHIVE = SAVED_ZIP
if not SOURCE_ARCHIVE.is_file():
    if not IN_COLAB:
        raise FileNotFoundError("Place the source ZIP beside this notebook or set SOURCE_ARCHIVE.")
    from google.colab import files
    uploaded = files.upload()
    matches = [Path(n) for n, data in uploaded.items() if hashlib.sha256(data).hexdigest() == EXPECTED_SHA]
    if len(matches) != 1:
        raise ValueError("Upload the matching source ZIP supplied with this notebook.")
    SOURCE_ARCHIVE = matches[0]
data = SOURCE_ARCHIVE.read_bytes()
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA:
    raise ValueError("Wrong source ZIP: checksum mismatch")
SAVED_ZIP.parent.mkdir(parents=True, exist_ok=True)
if not SAVED_ZIP.exists():
    SAVED_ZIP.write_bytes(data)
if hashlib.sha256(SAVED_ZIP.read_bytes()).hexdigest() != EXPECTED_SHA:
    raise ValueError("Saved source ZIP readback failed")
PROJECT = SCRATCH_ROOT / "release"
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
    for member in archive.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()) or member.is_dir():
            raise ValueError("Unsafe source ZIP entry")
        content = archive.read(member)
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Source changed: {target}; use a different empty scratch, keep results.")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
import uv
UV = str(uv.find_uv_bin())
CONTROLLER = SCRATCH_ROOT / "controller/bin/python"
if not CONTROLLER.exists():
    subprocess.run([UV, "venv", "--python", "3.10.12", str(CONTROLLER.parent.parent)], check=True)
lock = OUTPUT_ROOT / "_release/controller-resolved.lock"
if lock.is_file():
    subprocess.run([UV, "pip", "sync", "--python", str(CONTROLLER), str(lock)], check=True)
else:
    subprocess.run([UV, "pip", "install", "--python", str(CONTROLLER), "-r", str(PROJECT / "requirements.txt")], check=True)
    resolved = subprocess.check_output([UV, "pip", "freeze", "--python", str(CONTROLLER)], text=True)
    lock.write_text(resolved, encoding="utf-8")
    if lock.read_text(encoding="utf-8") != resolved:
        raise ValueError("Resolved environment lock readback failed")
ENV = os.environ.copy()
ENV.update(PYTHONPATH=os.pathsep.join([str(PROJECT / "src"), str(PROJECT / "production/src")]),
    PYTHONUNBUFFERED="1", OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="4",
    PATH=str(Path(UV).parent)+os.pathsep+ENV.get("PATH", ""),
    UV_PYTHON_INSTALL_DIR=str(SCRATCH_ROOT / "python"), CUBLAS_WORKSPACE_CONFIG=":4096:8")
CONFIG = SCRATCH_ROOT / "neural-settings.json"
CONFIG.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
print("Free SSD: %.1f GiB" % (shutil.disk_usage(SCRATCH_ROOT).free / 1024**3))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


## 2. System libraries

Restore the already reviewed MagickWand/GL dependencies for author implementations.


In [ ]:
# Only RoSteALS/SepWater need these author-import system libraries.
# Restore the setup used by RoSteALS_AUTHOR_FIX and RoSteALS_SepWater_RECOVERY.
# No model source, checkpoint, Python lock, payload or matching rule is changed.
import ctypes
import ctypes.util

def ensure_author_system_dependencies(models, in_colab):
    affected = {"rosteals", "sepwater"}
    selected = affected if models == "all" else affected.intersection(models)
    if not selected:
        return {}

    requirements = {
        "libmagickwand-dev": (
            ("MagickWand-6.Q16", "MagickWand-6.Q16HDRI", "MagickWand-7.Q16HDRI",
             "MagickWand-7.Q16", "MagickWand"), "MagickWandGenesis"),
        "libgl1": (("GL",), None),
    }

    def inspect_libraries():
        found = {}
        for package, (names, symbol) in requirements.items():
            for name in names:
                path = ctypes.util.find_library(name)
                if not path:
                    continue
                try:
                    library = ctypes.CDLL(path)
                    if symbol:
                        getattr(library, symbol)
                except (OSError, AttributeError):
                    continue
                found[package] = path
                break
        return found

    found = inspect_libraries()
    missing = [package for package in requirements if package not in found]
    if missing:
        install = ["apt-get", "-qq", "install", "-y", "--no-install-recommends", *missing]
        if not in_colab or not shutil.which("apt-get") or os.geteuid() != 0:
            raise RuntimeError(
                "RoSteALS/SepWater system libraries are missing. On Debian/Ubuntu run: "
                "sudo apt-get update && sudo " + " ".join(install)
                + ". Then rerun this setup cell; keep the existing output directory.")
        print("Installing missing system libraries for " + ", ".join(sorted(selected))
              + ": " + ", ".join(missing), flush=True)
        subprocess.run(["apt-get", "-qq", "update"], check=True, timeout=600)
        subprocess.run(install, check=True, timeout=600)
        found = inspect_libraries()
        missing = [package for package in requirements if package not in found]
        if missing:
            raise RuntimeError("System-library preflight failed after installation: "
                               + ", ".join(missing) + ". No model has been started.")
    print("RoSteALS/SepWater system-library preflight passed:", found, flush=True)
    return found

AUTHOR_SYSTEM_LIBRARIES = ensure_author_system_dependencies(MODELS, IN_COLAB)


## 3. Validate bank and freeze captions/seeds

No images are re-embedded. A missing bank/caption or changed release fails explicitly.


In [ ]:
subprocess.run([str(CONTROLLER), "-m", "wmbench_neural.runner", "validate", "--settings", str(CONFIG)], env=ENV, check=True)


## 4. Run

Clean preflight → GPU attack batches → saved ZIPs → unchanged decoder → durable scores → delete ZIPs. Repeat per pipeline and method. CPU input preparation/PNG encoding/Drive writes overlap GPU batches.

Temporary Drive space is needed for one method/pipeline stage (potentially tens of GiB). Logs and completed results survive disconnects. A failed call stops safely; restart the same cells.


In [ ]:
command = [str(CONTROLLER), "-m", "wmbench_neural.runner", "run", "--settings", str(CONFIG)]
process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", start_new_session=True)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError("Run stopped: inspect persistent logs. Completed rows and pending ZIPs are safe.")
except BaseException:
    if process.poll() is None:
        import signal
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
    raise


## 5. Report

Rebuilding reports performs no inference. Missing or safety-blocked generations are not decoding errors. The additional resize-only diagnostic is excluded from the 21-condition experiment.


In [ ]:
subprocess.run([str(CONTROLLER), "-m", "wmbench_neural.runner", "report", "--settings", str(CONFIG)], env=ENV, check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT_ROOT / "_reports/REPORT.md").read_text(encoding="utf-8")))
print("Aggregate report:", OUTPUT_ROOT / "_reports")
print("Per-method tables:", OUTPUT_ROOT / "<method>/tables")
print("Decoded bits and task provenance:", OUTPUT_ROOT / "<method>/records")


## Interpretation and limits

The canary has up to **60,900** evaluations; N=100 has up to **1,218,000**, fewer when a matched carrier is unavailable. A fixed batch improves throughput, not the scientific task count. This canary itself may span multiple sessions.

Static and CPU smoke checks are recorded in VALIDATION.md. Real CUDA pipelines still need this Colab acceptance run; no full-GPU success or runtime estimate is invented. Protocol source: bundled committed revision; live GitHub access was unavailable during preparation.
